# Flat-Volume Breakout Research Viewer

This notebook reviews research outputs from `research/alpha_flat_volume_breakout.py`.

It focuses on Grinold-style diagnostics:
- `IC` (rank information coefficient)
- `ICIR` (`mean(IC) / std(IC)`)
- `TC` proxy (rank corr between forecast scores and implemented research weights)
- `BR` proxy (effective breadth, `1/sum(w^2)`)
- Implied `IR` proxy (`IC * sqrt(BR) * TC`)
- Realized active `IR` from research portfolio active returns


In [5]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 200)
plt.style.use('seaborn-v0_8-whitegrid')


In [6]:
RESEARCH_ROOT = Path('../research/output')

run_dirs = []
if RESEARCH_ROOT.exists():
    for d in RESEARCH_ROOT.iterdir():
        if d.is_dir() and (d / 'summary.csv').exists():
            run_dirs.append(d)

run_dirs = sorted(run_dirs, key=lambda p: p.stat().st_mtime, reverse=True)
recent = run_dirs[:3]

recent_df = pd.DataFrame([
    {
        'run_tag': d.name,
        'modified_time': pd.to_datetime(d.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M:%S'),
        'has_by_date': (d / 'by_date.csv').exists(),
        'path': str(d),
    }
    for d in recent
])

display(Markdown('## Top 3 Recent Research Runs'))
display(recent_df if not recent_df.empty else pd.DataFrame({'message': ['No runs found under ../research/output']}))

DEFAULT_RUN_TAG = recent[0].name if recent else None
RUN_TAG = None  # Optional override, e.g. 'smoke_grinold_fvb'
SELECTED_RUN_TAG = RUN_TAG or DEFAULT_RUN_TAG

if SELECTED_RUN_TAG is None:
    raise FileNotFoundError('No research runs found. Run research/alpha_flat_volume_breakout.py first.')

RUN_DIR = RESEARCH_ROOT / SELECTED_RUN_TAG
print(f'Using run: {SELECTED_RUN_TAG}')
print(f'Run directory: {RUN_DIR.resolve()}')


## Top 3 Recent Research Runs

,run_tag,modified_time,has_by_date,path
0,sector_mom_top2_ab_biweekly,2026-03-01 19:38:07,False,../research/output/sector_mom_top2_ab_biweekly
1,sector_mom_top2_ab_monthly,2026-03-01 19:38:07,False,../research/output/sector_mom_top2_ab_monthly
2,sector_mom_top2_demo,2026-03-01 19:14:08,False,../research/output/sector_mom_top2_demo


Using run: sector_mom_top2_ab_biweekly
Run directory: /home/rockebull/proj/TradingAgents/research/output/sector_mom_top2_ab_biweekly


## Metric Interpretation

- `avg_rank_ic`: Average cross-sectional rank correlation between signal score and forward returns.
- `ic_ir`: Stability-adjusted IC (`avg_rank_ic / ic_std`).
- `avg_tc_proxy`: How aligned implementation weights are with alpha ranking.
- `avg_breadth_proxy`: Effective independent bets proxy.
- `implied_ir`: Grinold-style forecast IR proxy from `IC`, `TC`, and `BR`.
- `realized_active_ir`: Realized active-return IR from the research portfolio return stream.

Use these together: a signal can have weak IC but still produce positive realized IR if construction effects, concentration, or regime dynamics help realized returns.


In [7]:
summary_path = RUN_DIR / 'summary.csv'
params_path = RUN_DIR / 'params.json'
manifest_path = RUN_DIR / 'manifest.json'
by_date_path = RUN_DIR / 'by_date.csv'

summary = pd.read_csv(summary_path)
params = json.loads(params_path.read_text(encoding='utf-8')) if params_path.exists() else {}
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}
by_date = pd.read_csv(by_date_path, parse_dates=['date']) if by_date_path.exists() else None

display(Markdown('## Run Parameters'))
display(pd.DataFrame({'parameter': list(params.keys()), 'value': list(params.values())}))

display(Markdown('## Horizon Summary'))
display(summary)

if manifest:
    display(Markdown('## Manifest'))
    display(pd.DataFrame({'field': list(manifest.keys()), 'value': [manifest[k] for k in manifest.keys()]}))


## Run Parameters

,parameter,value
0,script,sector_momentum_top2_sector
1,start_date,2021-03-01
2,end_date,2026-02-28
3,universe_source,sp500_snapshot
4,universe_size,500
5,momentum_lookback_days,63
6,top_k_per_sector,2
7,rebalance_frequency,biweekly
8,transaction_cost_bps,5.0
9,fundamentals_dir,data/fundamentals/sp500


## Horizon Summary

,start_date,end_date,rebalance_frequency,momentum_lookback_days,top_k_per_sector,symbols_with_prices,symbols_with_sector,sectors_used,rebalance_points,avg_selected_names,average_turnover,transaction_cost_bps,total_return,cagr,annualized_volatility,sharpe,max_drawdown,sector_source,fundamentals_coverage
0,2021-03-01,2026-02-28,biweekly,63,2,482,481,11,249,22.0,0.628299,5.0,1.887798,0.237105,0.19378,1.195607,-0.284959,data/fundamentals/sp500/sp500_fundamentals_lat...,0.997925


## Manifest

,field,value
0,run_tag,sector_mom_top2_ab_biweekly
1,created_at_utc,2026-03-01T19:38:07.720453+00:00
2,command,research/sector_momentum_top2_sector.py --star...
3,python_version,3.13.12
4,git_head,86dbd7d9d6a60b9f961e9a71791c7c6b4813c855
5,config_json,None
6,artifacts,{'summary_csv': 'research/output/sector_mom_to...


In [8]:
plot_df = summary.copy().sort_values('horizon_days')
x = plot_df['horizon_days'].astype(str)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].bar(x, plot_df['implied_ir'], label='Implied IR', alpha=0.8)
axes[0].bar(x, plot_df['realized_active_ir'], label='Realized Active IR', alpha=0.8)
axes[0].set_title('Implied vs Realized IR by Horizon')
axes[0].set_xlabel('Horizon (days)')
axes[0].legend()

axes[1].plot(x, plot_df['avg_rank_ic'], marker='o', label='avg_rank_ic')
axes[1].plot(x, plot_df['ic_ir'], marker='o', label='ic_ir')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('IC Quality by Horizon')
axes[1].set_xlabel('Horizon (days)')
axes[1].legend()

axes[2].plot(x, plot_df['avg_tc_proxy'], marker='o', label='TC proxy')
axes[2].plot(x, plot_df['avg_breadth_proxy'], marker='o', label='Breadth proxy')
axes[2].set_title('Implementation and Breadth')
axes[2].set_xlabel('Horizon (days)')
axes[2].legend()

fig.suptitle(f'Run: {SELECTED_RUN_TAG}', y=1.05)
fig.tight_layout()
plt.show()


KeyError: 'horizon_days'

## Date-Level Diagnostics (if available)

If `by_date.csv` exists, the charts below show:
- Cumulative active return over time
- Rolling IC (20 observations)
- Daily implied IR component (`IC * sqrt(BR) * TC`)


In [ ]:
if by_date is None or by_date.empty:
    display(Markdown('No by-date diagnostics found for this run. Re-run research with `--save-by-date`.'))
else:
    d = by_date.copy().sort_values(['horizon_days', 'date'])

    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    for h, g in d.groupby('horizon_days'):
        g = g.sort_values('date')
        axes[0].plot(g['date'], g['active_return'].cumsum(), label=f'h={h}')
        axes[1].plot(g['date'], g['ic'].rolling(20, min_periods=5).mean(), label=f'h={h}')
        axes[2].plot(g['date'], g['implied_ir_component'], label=f'h={h}', alpha=0.9)

    axes[0].set_title('Cumulative Active Return by Horizon')
    axes[1].set_title('Rolling IC Mean (window=20)')
    axes[2].set_title('Daily Implied IR Component')
    axes[2].axhline(0, color='black', linewidth=1)

    for ax in axes:
        ax.legend(loc='best')

    fig.tight_layout()
    plt.show()


## Reproducibility

Use the saved run parameters to rerun exactly:


In [ ]:
cfg_path = Path('../research/configs/alpha_flat_volume_breakout_default.json')
cmd = (
    'PYTHONPATH=. conda run -n activepm python research/alpha_flat_volume_breakout.py ' +
    f'--config-json {cfg_path} --run-tag {SELECTED_RUN_TAG} --save-by-date'
)
print(cmd)
if manifest.get('command'):
    print('\nOriginal command from manifest:')
    print(manifest['command'])
